<a href="https://colab.research.google.com/github/AyaNabih7/Fine_Tune_BLOOM_for_Financial_QA/blob/main/Fine_Tune_BLOOM_for_Financial_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1: Setup - Installing Dependencies**

In [ ]:
print("--- Step 1: Installing Libraries ---")
!pip install -q transformers[torch] datasets accelerate bitsandbytes peft
print("✅ Libraries installed successfully!")

--- Step 1: Installing Libraries ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.5 MB/s eta 0:00:00
✅ Librar

# **Step 2: Imports and Google Drive Mount**




In [ ]:
print("--- Step 2: Importing Libraries and Mounting Google Drive ---")

import os
import torch
from google.colab import drive
from datasets import load_from_disk, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Mount Google Drive
# This will prompt you for authorization.
drive.mount('/content/drive')

print("✅ Drive mounted and libraries imported successfully!")

--- Step 2: Importing Libraries and Mounting Google Drive ---
Mounted at /content/drive
✅ Drive mounted and libraries imported successfully!


# **Step 3: Load and Prepare the Dataset**


In [ ]:
print("--- Step 3: Loading and Preparing the Dataset ---")

# Define the path to your dataset on Google Drive
dataset_path = '/content/drive/MyDrive/Colab Notebooks/Chatbot_Project/FIGQ_dataset'

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at path: {dataset_path}")

# Load the dataset from disk
full_dataset = load_from_disk(dataset_path)
print("✅ Dataset loaded successfully!")

# Create a formatting function to structure the data
def format_instruction(example):
    return f"Question: {example['question']}\nAnswer: {example['answer']}"

# Apply the formatting to the entire dataset
text_data = full_dataset['train'].map(lambda example: {'text': format_instruction(example)})

# Split the dataset into 80% for training and 20% for validation
train_test_split = text_data.train_test_split(test_size=0.2, seed=42)

# Rename the 'test' split to 'validation' for clarity with the Trainer
split_dataset = DatasetDict({
    'train': train_test_split['train'],
    'validation': train_test_split['test']
})

print("\n✅ Dataset successfully split into training and validation sets:")
print(split_dataset)


--- Step 3: Loading and Preparing the Dataset ---
✅ Dataset loaded successfully!

✅ Dataset successfully split into training and validation sets:
DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'text'],
        num_rows: 11608
    })
    validation: Dataset({
        features: ['question', 'answer', 'text'],
        num_rows: 2903
    })
})


# **Step 4: Load Model and Tokenizer**


In [ ]:
print("--- Step 4: Loading the BLOOM Model and Tokenizer ---")

model_name = "EleutherAI/gpt-neo-1.3B"

# Load the model with 8-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map='auto',
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set the padding token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Pad token set to EOS token.")

print(f"\n✅ Model '{model_name}' and tokenizer loaded successfully.")


--- Step 4: Loading the BLOOM Model and Tokenizer ---


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/5.31G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Pad token set to EOS token.

✅ Model 'EleutherAI/gpt-neo-1.3B' and tokenizer loaded successfully.


# **Step 5: Configure PEFT/LoRA for Efficient Fine-Tuning**


In [ ]:
print("--- Step 5: Configuring the Model with PEFT/LoRA ---")

# Prepare the quantized model for training
model = prepare_model_for_kbit_training(model)

# Define the LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print a summary of the trainable parameters
print("\n✅ Summary of trainable parameters after applying LoRA:")
model.print_trainable_parameters()


--- Step 5: Configuring the Model with PEFT/LoRA ---

✅ Summary of trainable parameters after applying LoRA:
trainable params: 3,145,728 || all params: 1,318,721,536 || trainable%: 0.2385


# **Step 6: Tokenize the Dataset**


In [ ]:
print("--- Step 6: Tokenizing the Dataset ---")

def tokenize_function(examples):
    # Tokenize the text. We apply padding and truncation to ensure
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    # For Causal LM, the labels are the same as the input IDs.
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

# Apply the tokenization function to the entire dataset
tokenized_datasets = split_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=split_dataset['train'].column_names
)

print("\n✅ Dataset successfully tokenized and padded.")

--- Step 6: Tokenizing the Dataset ---

✅ Dataset successfully tokenized and padded.


# **Step 7: Configure and Start the Training Process**


In [ ]:
# --- 7.1: Configure Training Arguments ---
print("--- Step 7.1: Configuring Training Arguments ---")

# Define the directory on Google Drive to save the final model
output_dir = "/content/drive/MyDrive/Colab Notebooks/Chatbot_Project/fiqa_gpt_neo_lora_finetuned"

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=500,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=500,
    fp16=True,
    report_to="none",
)

# --- 7.2: Initialize the Trainer ---
print("\n--- Step 7.2: Initializing the Trainer ---")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)




--- Step 7.1: Configuring Training Arguments ---

--- Step 7.2: Initializing the Trainer ---


In [ ]:
# --- 7.3: Start Training ---
print("\n--- Step 7.3: Starting the Fine-Tuning Process ---")
# This command will start the training. It may take a few hours.
trainer.train(resume_from_checkpoint=True)

# --- 7.4: Save the Final Model ---
print("\n--- Step 7.4: Saving the Final Model ---")
final_model_path = f"{output_dir}/final_model_v2"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"\n✅Training complete! ")
print(f"The final model (LoRA adapters) has been saved to:")
print(final_model_path)


--- Step 7.3: Starting the Fine-Tuning Process ---


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss
19000,1.252800,1.248764
19500,1.274500,1.248888
20000,1.281500,1.248915
20500,1.230100,1.248871
21000,1.203400,1.248721


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reen

In [ ]:
#!pip install -q transformers[torch] datasets accelerate bitsandbytes peft
from huggingface_hub import notebook_login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch


In [ ]:
notebook_login()
base_model_name = "EleutherAI/gpt-neo-1.3B"
lora_model_path = "/content/drive/MyDrive/Colab Notebooks/Chatbot_Project/fiqa_gpt_neo_lora_finetuned/final_model_v2"

base_model_for_merging = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map='auto',
)
model_to_merge = PeftModel.from_pretrained(base_model_for_merging, lora_model_path)
merged_model = model_to_merge.merge_and_unload()
tokenizer = AutoTokenizer.from_pretrained(base_model_name)




In [ ]:
hub_model_name = "gpt-neo-1.3B-fine-tuned-for-financial-QA-v2"
merged_model.push_to_hub(hub_model_name)
tokenizer.push_to_hub(hub_model_name)